# Analysis of the hottest heat periods in Germany and Europe

Select **DWD HOSTRADA** to work with the 20 hottest documented heat periods in Germany, or **Copernicus CERRA** to work with the 20 hottest documented heat periods in Europe.


In [1]:
import os
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display
from ipyleaflet import Map, Marker, ScaleControl, basemaps

from hostrada4py import hostradaPoint as hp
from hostrada4py import hostrada_HeatPeriodsGermany as heat_de
from hostrada4py import hostrada_heatPeriodsEurope as heat_eu

os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
# os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"


In [2]:
from hostrada4py import providerUI as provider_ui

provider_selector = provider_ui.create_provider_selector(
    globals(),
    initial="dwd",
    title="Select weather data source",
    show=True,
)

def _sync_notebook_provider(change=None):
    supported = {value for _, value in provider_ui.variable_options(provider_dropdown.value)}

    current_variable = globals().get("HOSTRADA_VAR")
    if current_variable is not None and current_variable not in supported:
        globals()["HOSTRADA_VAR"] = "tas" if "tas" in supported else next(iter(supported))

    selector = globals().get("variable_selector")
    if selector is not None and hasattr(selector, "options"):
        all_options = globals().get("_hostrada_all_variable_options")
        if all_options is None:
            all_options = list(selector.options)
            globals()["_hostrada_all_variable_options"] = all_options
        filtered = [
            option for option in all_options
            if (option[1] if isinstance(option, tuple) else option) in supported
        ]
        old_values = tuple(value for value in selector.value if value in supported)
        selector.value = ()
        selector.options = filtered
        available = [option[1] if isinstance(option, tuple) else option for option in filtered]
        selector.value = old_values or (("tas",) if "tas" in available else tuple(available[:1]))

provider_dropdown.observe(_sync_notebook_provider, names="value")
_sync_notebook_provider()


## Heat-period catalogues

The ranking is based on the highest documented station air temperature within each event's core period. It is not an official DWD or pan-European heat-wave intensity index.

- **DWD HOSTRADA:** 20 documented heat periods in Germany, reviewed on 4 August 2026.
- **Copernicus CERRA:** 20 documented European heat periods from September 1984 onward, matching the CERRA temporal coverage. The European catalogue was reviewed on 4 August 2026; summer 2026 events are excluded because they were not yet available in the CERRA archive.

Changing the provider automatically changes both the ranking table and the clickable period list.


In [3]:
ranking_heading = widgets.HTML()
ranking_output = widgets.Output()

def _active_heat_catalogue():
    return heat_de if provider_dropdown.value == "dwd" else heat_eu

def _active_catalogue_name():
    return "Germany / DWD HOSTRADA" if provider_dropdown.value == "dwd" else "Europe / Copernicus CERRA"

def _render_ranking(change=None):
    catalogue = _active_heat_catalogue()
    ranking_heading.value = f"<h3>20 hottest documented heat periods: {_active_catalogue_name()}</h3>"
    table = catalogue.heat_periods_dataframe(language="en")
    with ranking_output:
        clear_output(wait=True)
        display(
            table.style
            .format({
                "Maximum [°C]": "{:.1f}",
                "Latitude [°N]": "{:.4f}",
                "Longitude [°E]": "{:.4f}",
            })
            .hide(axis="index")
        )

provider_dropdown.observe(_render_ranking, names="value")
_render_ranking()
display(widgets.VBox([ranking_heading, ranking_output]))


## Select, locate, download and visualise a heat period

Click a period in the list. The Leaflet map immediately moves to the station location associated with the selected event and shows the event details in a marker popup. Then select **Load selected heat period** to retrieve the air-temperature time series, save it as a provider-specific CSV file and display it below the map.


In [4]:
def _catalogue_options():
    return _active_heat_catalogue().heat_period_options(language="en")

if hasattr(widgets, "Select"):
    heat_period_selector = widgets.Select(
        options=_catalogue_options(),
        value=1,
        description="Heat period:",
        rows=12,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="100%"),
    )
else:
    heat_period_selector = widgets.Dropdown(
        options=_catalogue_options(),
        value=1,
        description="Heat period:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="100%"),
    )

load_heat_period_button = widgets.Button(
    description="Load selected heat period",
    button_style="primary",
    icon="download",
    layout=widgets.Layout(width="100%"),
)
selection_details = widgets.HTML()
map_heading = widgets.HTML(value="<h3>Event location</h3>")
visualisation_output = widgets.Output()


def _selected_event_and_site():
    catalogue = _active_heat_catalogue()
    event = catalogue.get_heat_period(int(heat_period_selector.value))
    return catalogue, event, event["measurement_sites"][0]


def _map_zoom_for_selection(site):
    """Use a station-level view for DWD and a wider regional view for CERRA."""
    explicit_zoom = site.get("map_zoom")
    if explicit_zoom is not None:
        return int(explicit_zoom)
    return 8 if provider_dropdown.value == "dwd" else 7


def _map_popup_html(catalogue, event, site):
    country = site.get("country", "Germany")
    return (
        f"<div style='min-width:240px'>"
        f"<b>Rank {event['rank']}: {site['name']}</b><br>"
        f"{country}<br>"
        f"Core period: {catalogue.format_core_period(event, language='en')}<br>"
        f"Maximum: {event['maximum_temperature_c']:.1f} °C<br>"
        f"Coordinates: {site['latitude']:.4f} °N, {site['longitude']:.4f} °E"
        f"</div>"
    )


_initial_catalogue, _initial_event, _initial_site = _selected_event_and_site()
heat_event_marker = Marker(
    location=(_initial_site["latitude"], _initial_site["longitude"]),
    draggable=False,
    title=_initial_site["name"],
)
heat_event_marker.popup = widgets.HTML(
    value=_map_popup_html(_initial_catalogue, _initial_event, _initial_site)
)
heat_event_map = Map(
    center=(_initial_site["latitude"], _initial_site["longitude"]),
    zoom=_map_zoom_for_selection(_initial_site),
    basemap=basemaps.OpenStreetMap.Mapnik,
    scroll_wheel_zoom=True,
    layout=widgets.Layout(width="100%", height="430px"),
)
heat_event_map.add(heat_event_marker)
heat_event_map.add_control(ScaleControl(position="bottomleft"))


def _update_heat_event_map(catalogue, event, site):
    country = site.get("country", "Germany")
    lat = float(site["latitude"])
    lon = float(site["longitude"])
    heat_event_marker.location = (lat, lon)
    heat_event_marker.title = f"{site['name']}, {country}"
    heat_event_marker.popup = widgets.HTML(value=_map_popup_html(catalogue, event, site))
    heat_event_map.center = (lat, lon)
    heat_event_map.zoom = _map_zoom_for_selection(site)
    map_heading.value = (
        f"<h3>Event location</h3>"
        f"<p><b>{site['name']}, {country}</b> — click the marker for event details.</p>"
    )


def _update_selection_details(change=None):
    catalogue, event, site = _selected_event_and_site()
    country = site.get("country", "Germany")
    selection_details.value = (
        f"<b>Selection:</b> rank {event['rank']} – "
        f"{catalogue.format_core_period(event, language='en')}, "
        f"maximum {event['maximum_temperature_c']:.1f} °C, "
        f"station {site['name']}, {country} ({catalogue.format_status(event, language='en')})"
    )
    _update_heat_event_map(catalogue, event, site)


def _sync_heat_period_catalogue(change=None):
    new_options = _catalogue_options()
    heat_period_selector.options = new_options
    heat_period_selector.value = new_options[0][1]
    _update_selection_details()
    with visualisation_output:
        clear_output(wait=True)
        print(
            "The period list and map now match "
            + ("DWD HOSTRADA (Germany)." if provider_dropdown.value == "dwd" else "Copernicus CERRA (Europe).")
        )


def _load_selected_heat_period(_=None):
    global rank, start_UTC, end_UTC, location, meteostation, lon, lat, df, fn
    catalogue, event, site = _selected_event_and_site()
    country = site.get("country", "Germany")

    rank = event["rank"]
    start_UTC = event["core_period"]["start"]
    end_UTC = event["core_period"]["end"]
    location = site["name"]
    meteostation = site["source_url"]
    lon = site["longitude"]
    lat = site["latitude"]
    provider_name = provider_dropdown.value.upper()

    with visualisation_output:
        clear_output(wait=True)
        print(
            f"Loading rank {rank}: {catalogue.format_core_period(event, language='en')} – "
            f"{location}, {country} ({lat:.4f} °N, {lon:.4f} °E) via {provider_name}"
        )
        df = hp.extract_values_for_point(
            var="tas", lon=lon, lat=lat, start=start_UTC, end=end_UTC
        )
        fn = f"HOSTRADA_tas_{provider_dropdown.value}_rank_{rank:02d}.csv"
        df.to_csv(fn, index=False)
        print(f"Wrote {len(df)} rows to {fn}.")

        plot_df = df.copy()
        plot_df["time"] = pd.to_datetime(plot_df["time"])
        plot_df = plot_df.sort_values("time")
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(plot_df["time"], plot_df["tas"])
        ax.set_xlabel("Date and time")
        ax.set_ylabel("Air temperature [°C]")
        ax.set_title(
            f"{location}, {country} ({start_UTC} to {end_UTC})\n"
            f"Provider: {provider_name}; station reference: {meteostation}; "
            f"lon: {lon:.4f}°, lat: {lat:.4f}°"
        )
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M\n%Y"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.tick_params(axis="x", rotation=45)
        ax.grid(True)
        fig.tight_layout()
        plt.show()


heat_period_selector.observe(_update_selection_details, names="value")
provider_dropdown.observe(_sync_heat_period_catalogue, names="value")
load_heat_period_button.on_click(_load_selected_heat_period)
_update_selection_details()

controls_column = widgets.VBox(
    [heat_period_selector, selection_details, load_heat_period_button],
    layout=widgets.Layout(width="40%", min_width="340px"),
)
map_column = widgets.VBox(
    [map_heading, heat_event_map],
    layout=widgets.Layout(width="58%", min_width="420px"),
)

# A wrapping layout keeps the map readable on both wide and narrow notebook views.
display(
    widgets.VBox([
        widgets.HBox(
            [controls_column, map_column],
            layout=widgets.Layout(
                width="100%",
                display="flex",
                flex_flow="row wrap",
                align_items="stretch",
                justify_content="space-between",
            ),
        ),
        visualisation_output,
    ])
)
